In [2]:
import duckdb
import pandas as pd

con = duckdb.connect('../data/airbnb.db')
print("Setup complete")

Setup complete


## Cleaning Decisions (Documented)

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Price outliers (high) | Cap at $5,000/night | 600x median ($130) was clear error; $5K is above legitimate luxury |
| Price outliers (low) | Cap at $10/night | $1 is unrealistic for London market |
| NULL price (32,977 listings) | Preserve as NULL | Real data state, not error (correlates with NULL superhost) |
| NULL neighbourhood | Use neighbourhood_cleansed fallback | Geocoded version has 0% nulls, more accurate |
| NULL host_is_superhost | Preserve as 'unknown' | Different from 'f' (false) - not yet evaluated |
| Unavailable listings | Flag (is_active=FALSE) not delete | Keep for reference, filter in analysis |
| Property type (112 unique) | Group into 9 categories | Enables meaningful analysis |

Dropped "listings_clean" temporary view

In [11]:
con.execute("DROP TABLE IF EXISTS listings_clean")
con.execute("DROP TABLE IF EXISTS listings_final")
print("Cleaned up")

tables = con.execute("SHOW TABLES").fetchall()
print(f"Remaining: {[t[0] for t in tables]}")

Cleaned up
Remaining: ['dim_host', 'listings', 'reviews']


Standardized "price" column

In [12]:
# Task 1: Standardize price (strip $, commas, cast to DECIMAL)

con.execute("""
    CREATE TABLE listings_clean AS
    SELECT *,
        TRY_CAST(REPLACE(REPLACE(REPLACE(price, '$', ''), ',', ''), ' ', '') AS DECIMAL) as price_clean,
        TRY_CAST(REPLACE(host_response_rate, '%', '') AS INTEGER) as host_response_rate_clean,
        TRY_CAST(REPLACE(host_acceptance_rate, '%', '') AS INTEGER) as host_acceptance_rate_clean
    FROM listings
""")

# Verify
print("Sample price conversions:")
print(con.execute("SELECT price, price_clean FROM listings_clean LIMIT 5").df().to_string(index=False))
print(f"\n 715 prices with commas all converted successfully")
print(f" 31,876 'N/A' response rates → NULL")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sample price conversions:
  price  price_clean
 $59.00         59.0
$120.00        120.0
$493.00        493.0
$190.00        190.0
$140.00        140.0

 715 prices with commas all converted successfully
 31,876 'N/A' response rates → NULL


Verification of price standardizing step

In [13]:
print("=" * 60)
print("VERIFICATION: Price & Rate Conversions")
print("=" * 60)

# 1. Comma prices
comma_check = con.execute("""
    SELECT 
        COUNT(*) as total_with_commas,
        SUM(CASE WHEN price_clean IS NOT NULL THEN 1 ELSE 0 END) as successfully_converted
    FROM listings_clean
    WHERE price LIKE '%,%'
""").df()
print("\n Prices WITH commas:")
print(comma_check.to_string(index=False))

# 2. N/A in response rate
na_response = con.execute("""
    SELECT 
        COUNT(*) as total_na,
        SUM(CASE WHEN host_response_rate_clean IS NULL THEN 1 ELSE 0 END) as became_null
    FROM listings_clean
    WHERE host_response_rate = 'N/A'
""").df()
print("\n host_response_rate = 'N/A':")
print(na_response.to_string(index=False))

# 3. N/A in acceptance rate
na_acceptance = con.execute("""
    SELECT 
        COUNT(*) as total_na,
        SUM(CASE WHEN host_acceptance_rate_clean IS NULL THEN 1 ELSE 0 END) as became_null
    FROM listings_clean
    WHERE host_acceptance_rate = 'N/A'
""").df()
print("\n host_acceptance_rate = 'N/A':")
print(na_acceptance.to_string(index=False))

# 4. Sample: comma prices
print("\n" + "=" * 60)
print("SAMPLE: Comma prices converted")
print("=" * 60)
comma_samples = con.execute("""
    SELECT price as original, price_clean 
    FROM listings_clean 
    WHERE price LIKE '%,%'
    LIMIT 5
""").df()
print(comma_samples.to_string(index=False))

# 5. Sample: response rate conversions
print("\n" + "=" * 60)
print("SAMPLE: Response rate conversions")
print("=" * 60)
rate_samples = con.execute("""
    SELECT 
        host_response_rate as original, 
        host_response_rate_clean as cleaned
    FROM listings_clean 
    WHERE host_response_rate IS NOT NULL
    LIMIT 10
""").df()
print(rate_samples.to_string(index=False))

VERIFICATION: Price & Rate Conversions

 Prices WITH commas:
 total_with_commas  successfully_converted
               715                   715.0

 host_response_rate = 'N/A':
 total_na  became_null
    31876      31876.0

 host_acceptance_rate = 'N/A':
 total_na  became_null
    27221      27221.0

SAMPLE: Comma prices converted
 original  price_clean
$1,300.00       1300.0
$1,500.00       1500.0
$2,000.00       2000.0
$1,134.00       1134.0
$1,000.00       1000.0

SAMPLE: Response rate conversions
original  cleaned
    100%    100.0
     N/A      NaN
    100%    100.0
    100%    100.0
     N/A      NaN
    100%    100.0
    100%    100.0
    100%    100.0
     N/A      NaN
    100%    100.0


Identifing and capping extreme price-outliers

In [14]:
con.execute("""
    ALTER TABLE listings_clean ADD COLUMN price_final DECIMAL;
    ALTER TABLE listings_clean ADD COLUMN price_flag VARCHAR;
    
    UPDATE listings_clean SET 
        price_final = CASE 
            WHEN price_clean > 5000 THEN NULL
            WHEN price_clean < 10 THEN NULL
            ELSE price_clean
        END,
        price_flag = CASE 
            WHEN price_clean > 5000 THEN 'EXTREME_HIGH'
            WHEN price_clean < 10 THEN 'EXTREME_LOW'
            ELSE 'NORMAL'
        END
    WHERE 1=1;
""")

# Show flag distribution
print(" Price flag distribution:")
print(con.execute("""
    SELECT price_flag, COUNT(*) as count
    FROM listings_clean GROUP BY price_flag
""").df().to_string(index=False))

 Price flag distribution:
  price_flag  count
EXTREME_HIGH     47
 EXTREME_LOW      7
      NORMAL  96128


Date field validation

In [15]:
date_columns = con.execute("""
    SELECT column_name, column_type
    FROM (SUMMARIZE listings_clean)
    WHERE column_type = 'DATE'
""").df()
print("DATE COLUMNS:")
print(date_columns.to_string(index=False))

date_ranges = con.execute("""
    SELECT 
        MIN(last_scraped) as min_scrape, MAX(last_scraped) as max_scrape,
        MIN(first_review) as min_first_review, MAX(first_review) as max_first_review
    FROM listings_clean
""").df()
print("\n Date ranges:")
print(date_ranges.to_string(index=False))

DATE COLUMNS:
          column_name column_type
         last_scraped        DATE
           host_since        DATE
calendar_last_scraped        DATE
         first_review        DATE
          last_review        DATE

 Date ranges:
min_scrape max_scrape min_first_review max_first_review
2024-09-06 2024-09-11       2009-12-21       2024-09-06


Normalized text fields & handling missing values

In [16]:
# Add neighbourhood_final (uses cleansed as fallback)
con.execute("""
    ALTER TABLE listings_clean ADD COLUMN neighbourhood_final VARCHAR;
    UPDATE listings_clean 
    SET neighbourhood_final = COALESCE(neighbourhood, neighbourhood_cleansed, 'Unknown');
""")

# Add superhost_status (NULL → 'unknown')
con.execute("""
    ALTER TABLE listings_clean ADD COLUMN superhost_status VARCHAR;
    UPDATE listings_clean 
    SET superhost_status = CASE 
        WHEN host_is_superhost IS NULL THEN 'unknown'
        WHEN host_is_superhost = TRUE THEN 't'
        ELSE 'f'
    END;
""")

# Add property_category (group 112 types into 9)
con.execute("""
    ALTER TABLE listings_clean ADD COLUMN property_category VARCHAR;
    UPDATE listings_clean SET property_category = CASE
        WHEN property_type ILIKE '%apartment%' OR property_type ILIKE '%flat%' OR property_type ILIKE '%condo%' THEN 'Apartment'
        WHEN property_type ILIKE '%house%' OR property_type ILIKE '%home%' OR property_type ILIKE '%bungalow%' OR property_type ILIKE '%cottage%' THEN 'House'
        WHEN property_type ILIKE '%hotel%' OR property_type ILIKE '%boutique%' THEN 'Hotel'
        WHEN property_type ILIKE '%loft%' THEN 'Loft'
        WHEN property_type ILIKE '%townhouse%' THEN 'Townhouse'
        WHEN property_type ILIKE '%villa%' THEN 'Villa'
        WHEN property_type ILIKE '%cabin%' OR property_type ILIKE '%chalet%' THEN 'Cabin/Chalet'
        WHEN property_type ILIKE '%hostel%' THEN 'Hostel'
        WHEN property_type ILIKE '%guesthouse%' OR property_type ILIKE '%bed and breakfast%' OR property_type ILIKE '%b&b%' THEN 'Guesthouse'
        ELSE 'Other'
    END;
""")

# Verify
print(" Neighbourhood handling:")
print(con.execute("""
    SELECT 
        SUM(CASE WHEN neighbourhood IS NULL THEN 1 ELSE 0 END) as original_nulls,
        SUM(CASE WHEN neighbourhood_final = 'Unknown' THEN 1 ELSE 0 END) as still_unknown
    FROM listings_clean
""").df().to_string(index=False))

print("\n Property categories:")
print(con.execute("""
    SELECT property_category, COUNT(*) as count
    FROM listings_clean GROUP BY property_category ORDER BY count DESC
""").df().to_string(index=False))

 Neighbourhood handling:
 original_nulls  still_unknown
        50520.0            0.0

 Property categories:
property_category  count
            Other  55205
            House  24096
        Apartment  14445
            Hotel   1159
             Loft    540
       Guesthouse    521
           Hostel    113
     Cabin/Chalet     53
            Villa     50


Flag invalid records

In [17]:
con.execute("""
    ALTER TABLE listings_clean ADD COLUMN is_active BOOLEAN DEFAULT TRUE;
    UPDATE listings_clean 
    SET is_active = FALSE
    WHERE name ILIKE '%no longer available%' OR name ILIKE '%unavailable%';
""")

print("Unavailable listings flagged:")
print(con.execute("""
    SELECT 
        SUM(CASE WHEN is_active = FALSE THEN 1 ELSE 0 END) as flagged,
        SUM(CASE WHEN is_active = TRUE THEN 1 ELSE 0 END) as active
    FROM listings_clean
""").df().to_string(index=False))

Unavailable listings flagged:
 flagged  active
    12.0 96170.0


Standardized geography

In [21]:
con.execute("""
    ALTER TABLE listings_clean ADD COLUMN latitude_clean DOUBLE;
    ALTER TABLE listings_clean ADD COLUMN longitude_clean DOUBLE;
""")

In [22]:
con.execute("""
    UPDATE listings_clean 
    SET latitude_clean = ROUND(latitude, 5),
        longitude_clean = ROUND(longitude, 5);
""")

print("Coordinates standardized (5 decimal precision):")
print(con.execute("""
    SELECT 
        MIN(latitude_clean) as min_lat, MAX(latitude_clean) as max_lat,
        MIN(longitude_clean) as min_lon, MAX(longitude_clean) as max_lon
    FROM listings_clean WHERE is_active = TRUE
""").df().to_string(index=False))

Coordinates standardized (5 decimal precision):
 min_lat  max_lat  min_lon  max_lon
51.29594 51.68164  -0.4978  0.29573


Final cleaned table

In [23]:
con.execute("""
    CREATE TABLE listings_final AS
    SELECT 
        id, name, description,
        host_id, host_name, host_since, host_location,
        neighbourhood_cleansed as neighbourhood,
        neighbourhood_final,
        latitude_clean as latitude,
        longitude_clean as longitude,
        room_type, property_type, property_category,
        price_final as price,
        price_flag,
        minimum_nights, maximum_nights, availability_365,
        number_of_reviews, number_of_reviews_ltm, reviews_per_month,
        first_review, last_review,
        review_scores_rating, review_scores_accuracy, review_scores_cleanliness,
        review_scores_checkin, review_scores_communication,
        review_scores_location, review_scores_value,
        host_is_superhost, superhost_status,
        host_response_rate_clean as host_response_rate,
        host_acceptance_rate_clean as host_acceptance_rate,
        calculated_host_listings_count,
        instant_bookable,
        is_active
    FROM listings_clean
""")

print("listings_final created")

listings_final created


Validation as of quality report expectations

In [3]:
print("=" * 60)
print("VALIDATION")
print("=" * 60)

total = con.execute("SELECT COUNT(*) FROM listings_final").fetchone()[0]
with_price = con.execute("SELECT COUNT(*) FROM listings_final WHERE price IS NOT NULL").fetchone()[0]
active = con.execute("SELECT COUNT(*) FROM listings_final WHERE is_active = TRUE").fetchone()[0]
no_price = con.execute("SELECT COUNT(*) FROM listings_final WHERE price IS NOT NULL AND price_flag = 'NO_PRICE'").fetchone()[0] + con.execute("SELECT COUNT(*) FROM listings_final WHERE price IS NULL AND price_flag = 'NO_PRICE'").fetchone()[0]

# Recalculate the NO_PRICE flag properly
con.execute("""
    UPDATE listings_final SET price_flag = 'NO_PRICE' WHERE price IS NULL
""")
no_price = con.execute("SELECT COUNT(*) FROM listings_final WHERE price_flag = 'NO_PRICE'").fetchone()[0]

print(f"\n Total: {total:,} (expected: 96,182)")
print(f" Active: {active:,} (expected: 96,170)")
print(f" With valid price: {with_price:,} (expected: ~63,151)")
print(f" NULL price (total): {no_price:,} (expected: 33,031 = 32,977 original + 54 capped)")

VALIDATION

 Total: 96,182 (expected: 96,182)
 Active: 96,170 (expected: 96,170)
 With valid price: 63,151 (expected: ~63,151)
 NULL price (total): 33,031 (expected: 33,031 = 32,977 original + 54 capped)


In [4]:
con.close()
print("Connection closed!")

Connection closed!
